## LAB 02 — Entendendo os dados com o Assistant (SupportIQ)

1. Rode o **LAB 01** antes (tabela Delta no mesmo `catalog_name` / `schema_name`).
2. Confira os widgets.
3. Cluster ou serverless → **Run All**.
4. Use o Assistant (`Ctrl + I`) só com as colunas do CSV. Colunas com espaço: backtick.
5. Não pergunte sobre faturas, CNPJ, bairros, `Topic_group` ou Dataset 2 — **não existem** neste pacote.

In [ ]:
def widget_or_default(name: str, default: str) -> str:
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "supportiq")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")
print(f"{catalog_name}.{schema_name}")

In [ ]:
display(spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}"))

### Passo 1 — Perguntas abertas (Databricks Assistant)

1. Copie uma pergunta.
2. Clique no Assistant ao lado da célula SQL ou use `Ctrl + I`.
3. Peça SQL que use **somente** as 17 colunas de `customer_support_tickets`.
4. Execute o código sugerido.

`Time to Resolution` é timestamp. Horas de fila = diferença para `First Response Time`, só onde `Ticket Status` = Closed.

## Perguntas (colunas reais do CSV)

- Quantos tickets por `Ticket Status`, `Ticket Type`, `Ticket Priority` e `Ticket Channel`?
- Em Closed: mediana e p95 de horas (`Time to Resolution` − `First Response Time`) por canal × prioridade × tipo. Qual combinação trava?
- CSAT médio (`Customer Satisfaction Rating`) por canal, tipo e prioridade (só Closed).
- Horas totais e médias de resolução em Closed. Quanto está em Critical / High?
- Tickets Open vs Pending: há `First Response Time`? Há CSAT?
- Os 10 `Product Purchased` com mais tickets Closed e pior CSAT médio.
- `Ticket Subject` mais frequente em Critical ainda Open.
- Correlação (ou pelo menos cruzamento) entre horas de resolução e CSAT — não assumir causalidade.

In [ ]:
display(
    spark.sql(
        """
SELECT
  COUNT(*) AS tickets,
  SUM(CASE WHEN `Ticket Status` = 'Closed' THEN 1 ELSE 0 END) AS closed,
  SUM(CASE WHEN `Ticket Status` = 'Open' THEN 1 ELSE 0 END) AS open,
  SUM(CASE WHEN `Ticket Status` = 'Pending Customer Response' THEN 1 ELSE 0 END) AS pending,
  ROUND(AVG(CASE WHEN `Ticket Status` = 'Closed' THEN `Customer Satisfaction Rating` END), 3) AS csat_closed,
  ROUND(
    AVG(
      CASE
        WHEN `Ticket Status` = 'Closed'
          AND `Time to Resolution` IS NOT NULL
          AND `First Response Time` IS NOT NULL
        THEN (unix_timestamp(`Time to Resolution`) - unix_timestamp(`First Response Time`)) / 3600.0
      END
    ),
    2
  ) AS avg_resolution_hours_closed
FROM customer_support_tickets
"""
    )
)

In [ ]:
display(
    spark.sql(
        """
SELECT
  `Ticket Channel`,
  `Ticket Priority`,
  `Ticket Type`,
  COUNT(*) AS closed_tickets,
  ROUND(
    AVG(
      (unix_timestamp(`Time to Resolution`) - unix_timestamp(`First Response Time`)) / 3600.0
    ),
    2
  ) AS avg_hours,
  ROUND(
    PERCENTILE_APPROX(
      (unix_timestamp(`Time to Resolution`) - unix_timestamp(`First Response Time`)) / 3600.0,
      0.95
    ),
    2
  ) AS p95_hours
FROM customer_support_tickets
WHERE `Ticket Status` = 'Closed'
  AND `Time to Resolution` IS NOT NULL
  AND `First Response Time` IS NOT NULL
GROUP BY `Ticket Channel`, `Ticket Priority`, `Ticket Type`
ORDER BY p95_hours DESC
LIMIT 20
"""
    )
)

In [ ]:
display(
    spark.sql(
        """
SELECT
  `Ticket Status`,
  COUNT(*) AS tickets,
  SUM(CASE WHEN `First Response Time` IS NULL THEN 1 ELSE 0 END) AS missing_frt,
  SUM(CASE WHEN `Time to Resolution` IS NULL THEN 1 ELSE 0 END) AS missing_ttr,
  SUM(CASE WHEN `Customer Satisfaction Rating` IS NULL THEN 1 ELSE 0 END) AS missing_csat,
  SUM(CASE WHEN Resolution IS NULL OR Resolution = '' THEN 1 ELSE 0 END) AS missing_resolution
FROM customer_support_tickets
GROUP BY `Ticket Status`
ORDER BY tickets DESC
"""
    )
)

### Bônus — prompt para o Assistant

> Estude as 17 colunas de `customer_support_tickets`. Não invente `Topic_group` nem Dataset 2. Use Closed para CSAT e para horas (TTR − FRT). Liste gargalos por `Ticket Channel` × `Ticket Priority` × `Ticket Type`, o que move CSAT, e estime horas recuperáveis. Marque o que um humano ainda precisa ver (Critical / Cancellation / Refund com texto longo). Não exponha `Customer Email`.

### Genie

Sala publicada: [lab-002-support-genie](https://dbc-0dea39b3-be1e.cloud.databricks.com/genie/rooms/01f1b6165bac181a9836cde73e8e7556?o=7474644277755368) (`01f1b6165bac181a9836cde73e8e7556`).

O chatbot local (`e2e-chatbot-app-next`) abre esta sala em `/room/support`.

17 colunas. Horas = TTR - FRT so Closed. Sem Topic_group.
